# Clustering (et exploitation des données géographiques)

Dans cette partie nous exploitons les données géographiques et réalisons le clustering. 

Nous exportons des statistiques par commune pour croisement avec les données de vote. 

In [2]:
from BDTopo_fonctions import load_gpkg
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg")

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Chargement réussi (316142 lignes)


In [ ]:
#PB DANS GDF : ON A DES OBS EN DOUBLE (VIENT SUREMENT DE TEMP_STOCK2013 ??)

In [97]:
import importlib
import clustering_fonctions
importlib.reload(clustering_fonctions)
from clustering_fonctions import run_dbscan_parallele
temp=run_dbscan_parallele(gdf,400,6)

In [108]:
subset = temp[(temp["cluster_id_2025"] == -1) & (temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>6000)].copy()

# Reprojection vers WGS84 (latitude / longitude)
subset = subset.to_crs(epsg=4326)

# Calcul du centroïde
subset["centroid"] = subset.geometry.centroid
subset["lat"] = subset.centroid.y
subset["lon"] = subset.centroid.x

subset.sample(10)

/tmp/ipykernel_2190/214205017.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["centroid"] = subset.geometry.centroid
/tmp/ipykernel_2190/214205017.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["lat"] = subset.centroid.y
/tmp/ipykernel_2190/214205017.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["lon"] = subset.centroid.x


,Base,DEP_CODE,COMM,DATE_REELLE_AUTORISATION,DATE_REELLE_DOC,DATE_REELLE_DAACT,SURF_CREEE,I_EXTENSION,DESTINATION_PRINCIPALE,ID,...,cluster_id_2019,cluster_id_2020,cluster_id_2021,cluster_id_2022,cluster_id_2023,cluster_id_2024,cluster_id_2025,centroid,lat,lon
201397,Sitadel,62,62502,02/10/2018,22/10/2018,28/10/2019,14031.0,False,8,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (2.64384 50.61422),50.614216,2.643843
73182,Sitadel,28,28303,30/09/2016,10/10/2016,None,14868.0,False,8,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (1.85767 48.08744),48.087441,1.857672
155616,Sitadel,53,53125,30/07/2020,30/11/2020,07/03/2022,10733.0,False,6,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (-1.04353 48.48746),48.487462,-1.043530
133295,Sitadel,45,45024,24/11/2023,31/07/2024,None,10062.0,False,6,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (1.67005 47.81033),47.810333,1.670050
249684,Sitadel,76,76497,01/07/2014,15/10/2014,None,8000.0,False,8,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (1.01494 49.38866),49.388660,1.014940
165458,Sitadel,56,56146,06/02/2019,07/02/2019,01/09/2021,66174.0,False,8,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (-2.92569 48.1102),48.110198,-2.925685
221798,Sitadel,68,68374,24/11/2017,None,None,6239.0,False,4,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (7.29354 48.07408),48.074078,7.293535
184714,Sitadel,59,59509,07/02/2019,None,None,6348.0,False,8,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (3.10284 50.41838),50.418379,3.102840
165854,Sitadel,56,56126,07/07/2023,04/09/2023,None,6080.0,False,6,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (-2.32502 47.5315),47.531497,-2.325022
228256,Sitadel,69,69275,20/05/2016,23/07/2018,29/01/2021,23001.0,False,4,None,...,-1,-1,-1,-1,-1.0,-1,-1,POINT (4.95836 45.76836),45.768363,4.958356


In [106]:
# Cluster_id présents dans Sitadel
sitadel_ids = temp.loc[temp["Base"]=="Sitadel", "cluster_id_2014"].unique()

# Comptage des occurrences de tous les bâtiments pour ces cluster_id
vc = temp["cluster_id_2025"].value_counts()

# Filtrer uniquement les clusters présents dans Sitadel
vc_sitadel = vc[vc.index.isin(sitadel_ids)]

# Nombre de clusters de taille 
vc_sitadel[vc_sitadel.index > 0]

cluster_id_2025
9300002    1889
5900063    1384
6900002    1371
9200004    1097
5900007     983
           ... 
9100075       6
5100089       6
5200036       6
4900105       6
4100043       6
Name: count, Length: 809, dtype: int64

In [110]:
# CLUSTER TOTAL, TOUS BAT EXISTANTS OU AUTORISES, ANNEE PAR ANNEE 
import pandas as pd
temp_sorted = temp.sort_values("Annee_REF")
results = []
for annee in sorted(temp_sorted["Annee_REF"].unique()):
    mask = temp_sorted["Annee_REF"] <= annee  # toutes les années <= annee
    cluster_col = temp_sorted.loc[mask, "cluster_id"]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    results.append({
        "Annee": annee,
        "pct_cluster_cumule": pct_cluster
    })
df_cumule = pd.DataFrame(results)
print(df_cumule)

    Annee  pct_cluster_cumule
0    2013           55.834452
1    2014           55.636189
2    2015           55.392046
3    2016           55.141323
4    2017           55.059186
5    2018           55.067007
6    2019           57.147890
7    2020           57.114960
8    2021           57.232919
9    2022           57.403139
10   2023           57.500650
11   2024           57.552008
12   2025           57.622208


In [116]:
temp[
    (temp["Base"] == "Sitadel") & (temp["SURF_CREEE"] > seuil)
].groupby("Annee_REF").size()

Annee_REF
2014     67
2015     80
2016    104
2017     97
2018    105
2019     92
2020     69
2021     88
2022    100
2023     85
2024     67
2025     47
dtype: int64

In [117]:
#PC QUI EN 2025 SONT DANS UN CLUSTER
seuil=10000
temp[(temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>seuil)].groupby("Annee_REF")["cluster_id"].apply(lambda x: (x > 0).sum() / len(x) * 100)

Annee_REF
2014    82.089552
2015    78.750000
2016    64.423077
2017    72.164948
2018    72.380952
2019    73.913043
2020    71.014493
2021    64.772727
2022    74.000000
2023    72.941176
2024    82.089552
2025    87.234043
Name: cluster_id, dtype: float64

In [114]:
#PC QUI SONT DANS UN CLUSTER AU MOMENT DE AUTORISATION 

temp_sitadel = temp[(temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>seuil)]

for annee in sorted(temp_sitadel["Annee_REF"].unique()):
    col = f"cluster_id_{annee}"
    mask = temp_sitadel["Annee_REF"] == annee
    cluster_col = temp_sitadel.loc[mask, col]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    print(f"Année {annee}: {pct_cluster:.1f}% des bâtiments clusterisés")

Année 2014: 59.7% des bâtiments clusterisés
Année 2015: 55.0% des bâtiments clusterisés
Année 2016: 52.9% des bâtiments clusterisés
Année 2017: 59.8% des bâtiments clusterisés
Année 2018: 58.1% des bâtiments clusterisés
Année 2019: 64.1% des bâtiments clusterisés
Année 2020: 63.8% des bâtiments clusterisés
Année 2021: 61.4% des bâtiments clusterisés
Année 2022: 70.0% des bâtiments clusterisés
Année 2023: 70.6% des bâtiments clusterisés
Année 2024: 82.1% des bâtiments clusterisés
Année 2025: 87.2% des bâtiments clusterisés


In [118]:
annees = sorted(temp["Annee_REF"].unique())
result_list = []
for annee in annees:
    col = f"cluster_id_{annee}"
    if col in temp.columns:
        # compter les clusters uniques non-négatifs
        nb_clusters = temp[col][temp[col] > 0].nunique()
        result_list.append({"Annee_REF": annee, "nb_clusters": nb_clusters})

df_clusters_par_annee = pd.DataFrame(result_list)
print(df_clusters_par_annee)

    Annee_REF  nb_clusters
0        2013         5236
1        2014         5424
2        2015         5591
3        2016         5695
4        2017         5841
5        2018         5927
6        2019         7096
7        2020         7273
8        2021         7338
9        2022         7444
10       2023         7517
11       2024         7592
12       2025         7679
